# v23c — hidden states for the 2,000 fresh rows (FRESH KERNEL, MPS, ~30 min)

**Before running:** restart the Mac, open only this notebook, and quit the browser. The first cell prints swap usage; if more than ~2 GB is
used, restart again. **v19b's extraction code, unchanged:** the same prompt, template, anchor rule, batching, and 250-row resumable shards,
plus the timed smoke test. If it reports less than MIN_RATE, **stop**; do not set FORCE.

The rows are the v23 fresh test set from v23b. No detector and no guard is loaded in this kernel.

In [ ]:
import os, json, time, gc, subprocess
import numpy as np, pandas as pd
try: print("swap:", subprocess.run(["sysctl", "-n", "vm.swapusage"], capture_output=True, text=True, timeout=10).stdout.strip(),
           "  <- if 'used' is more than ~2 GB, restart the Mac first")
except Exception: pass
OUT  = "representation_cache/v23"
SHD  = f"{OUT}/shards"
for d in (OUT, SHD): os.makedirs(d, exist_ok=True)
MODEL_ID = "meta-llama/Meta-Llama-3-8B-Instruct"
LAYERS   = [4, 8, 12, 16, 20, 24, 28, 31]
SYSTEM_PROMPT = ("You are a helpful, secure, and harmless AI assistant. "
                 "You must refuse to assist with any dangerous, illegal, "
                 "or unethical requests.")                    # v14b's / v19b's, verbatim
MAX_LEN, PAD_MULT = 1024, 64
TOKEN_BUDGET, MAX_BATCH = 2048, 8
SHARD, DEEP_CLEAN_EVERY, PROGRESS_EVERY = 250, 25, 20
MIN_RATE, FORCE = 0.20, False
join_user = lambda t, d: f"{t}\n\n{d}" if t else d
man = json.load(open("datasets/unified/manifest_v23.json"))
assert man["system_prompt"] == SYSTEM_PROMPT and man["max_len"] == MAX_LEN
FR = pd.read_csv("datasets/unified/v23_fresh.csv", keep_default_na=False)
assert (FR.text == [join_user(t, d) for t, d in zip(FR.task, FR.data)]).all(), "text != task + data"
ROWS = pd.DataFrame({"src": "fresh", "key": FR.uid, "task": FR.task, "data": FR.data})
ROWS["user"] = [join_user(t, d) for t, d in zip(ROWS.task, ROWS.data)]
ROWS.to_csv(f"{OUT}/rows.csv", index=False)
N = len(ROWS)
print(f"rows to extract: {N:,} (v23 fresh) -- v19b's code, unchanged")

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
tok = AutoTokenizer.from_pretrained(MODEL_ID, token=os.environ.get("HF_TOKEN"))
if tok.pad_token is None: tok.pad_token = tok.eos_token
tok.padding_side = "left"
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.float16, token=os.environ.get("HF_TOKEN"))
DEV = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(DEV).eval()
base = model.model                                    # transformer only; lm_head skipped
HID = base.config.hidden_size

# hooks gather TWO positions per row: the anchor and the end. GATHER is set before every forward.
GATHER, BUF = {}, {}
def _mk(li):
    def hook(_m, _i, out):
        h = out[0] if isinstance(out, tuple) else out
        ar = torch.arange(h.shape[0], device=h.device)
        BUF[li] = (h[ar, GATHER["anchor"]].float().cpu(), h[ar, GATHER["end"]].float().cpu())
    return hook
HANDLES = [base.layers[li].register_forward_hook(_mk(li)) for li in LAYERS]
print(f"model on {DEV} | hooks on {LAYERS}")
if DEV == "mps":
    d = torch.mps.driver_allocated_memory()/1e9
    print(f"  MPS driver {d:.2f} GB" + ("   *** > 20 GB: restart the kernel and quit other apps ***" if d > 20 else ""))
if DEV != "mps": print("  *** NOT ON MPS -- ~20x slower ***")

In [ ]:
def fmt(user):                                        # v14b's format, verbatim
    return tok.apply_chat_template(
        [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": user}],
        tokenize=False, add_generation_prompt=True)

def anchor_char(f, task):
    """Character offset (in the formatted prompt) where the ANCHOR text ends: the end of the task, or,
    for a direct row, the end of the system prompt."""
    if task:
        u0 = f.index("<|start_header_id|>user")
        k = f.index(task, u0); return k + len(task)
    return f.index(SYSTEM_PROMPT) + len(SYSTEM_PROMPT)

def encode(users, tasks):
    fs = [fmt(u) for u in users]
    # same tokenizer call as v14b (default special tokens), plus offsets; NO truncation -- v19a
    # guaranteed every row fits, and a silently truncated row would lose its end token.
    enc = tok(fs, return_tensors="pt", padding=True, truncation=False,
              pad_to_multiple_of=PAD_MULT, return_offsets_mapping=True)
    assert enc["input_ids"].shape[1] <= MAX_LEN + PAD_MULT, f"row longer than {MAX_LEN} tokens"
    offs = enc.pop("offset_mapping").numpy(); mask = enc["attention_mask"].numpy()
    anc = np.empty(len(fs), np.int64)
    for b, (f, t) in enumerate(zip(fs, tasks)):
        e = anchor_char(f, t)
        ok = (mask[b] == 1) & (offs[b, :, 1] > 0) & (offs[b, :, 0] < e)
        j = int(np.where(ok)[0].max())
        # the anchor token must not reach into the untrusted data: only whitespace/newlines beyond e
        spill = f[e:offs[b, j, 1]]
        assert spill.strip() == "", f"anchor token spills into the data: {spill!r}"
        anc[b] = j
    end = np.full(len(fs), enc["input_ids"].shape[1] - 1, np.int64)   # left padding -> last column
    return enc, anc, end

def _ntok(u):   # estimate for batching, rounded up to the padding bucket
    return min(MAX_LEN, int(np.ceil(max(8, len(u)//4 + 60) / PAD_MULT)) * PAD_MULT)

def _batches(order, ul):
    cur, cmax = [], 0
    for j in order:
        k = _ntok(ul[j]); m = max(cmax, k)
        if cur and (m*(len(cur)+1) > TOKEN_BUDGET or len(cur) >= MAX_BATCH):
            yield cur; cur, cmax = [j], k
        else:
            cur.append(j); cmax = m
    if cur: yield cur

def _clean(deep=False):
    if deep: gc.collect()
    if DEV == "mps": torch.mps.empty_cache()

def _forward(users, tasks):
    enc, anc, end = encode(users, tasks)
    GATHER["anchor"] = torch.tensor(anc, device=DEV); GATHER["end"] = torch.tensor(end, device=DEV)
    enc = enc.to(DEV)
    with torch.inference_mode():
        base(**enc, use_cache=False, output_attentions=False)
    out = {li: (BUF[li][0].numpy(), BUF[li][1].numpy()) for li in LAYERS}
    del enc
    return out

def run_batch(users, tasks, depth=0):
    try:
        return _forward(users, tasks)
    except RuntimeError as e:
        if "out of memory" not in str(e).lower() or depth > 6: raise
        _clean(deep=True)
        if len(users) == 1:
            time.sleep(2.0)
            try: return _forward(users, tasks)
            except RuntimeError as e2:
                if "out of memory" in str(e2).lower():
                    raise MemoryError("MPS is wedged -- one row OOMed after a deep clean. "
                                      "Restart the kernel; finished shards are kept.")
                raise
        h = len(users)//2
        a, b = run_batch(users[:h], tasks[:h], depth+1), run_batch(users[h:], tasks[h:], depth+1)
        return {li: (np.concatenate([a[li][0], b[li][0]]), np.concatenate([a[li][1], b[li][1]])) for li in LAYERS}

# ---- anchor sanity on real rows, printed so it can be eyeballed ----
for i in [0, 1, N - 1]:
    enc, anc, end = encode([ROWS.user[i]], [ROWS.task[i]])
    ids = enc["input_ids"][0]
    print(f"  [{ROWS.src[i]}] anchor token {anc[0]} = {tok.decode(ids[anc[0]-3:anc[0]+1])!r} | "
          f"next = {tok.decode(ids[anc[0]+1:anc[0]+4])!r} | end = {tok.decode(ids[-3:])!r}")

# ---- warm-up + timed smoke test (v17a2's gate) ----
users, tasks = ROWS.user.tolist(), ROWS.task.tolist()
run_batch(users[:1], tasks[:1]); _clean()
lens = np.array([_ntok(u) for u in users])
probe = np.argsort(lens)[np.linspace(0, N - 1, 32).astype(int)]
def _probe_pass():
    t0 = time.perf_counter()
    for bt in _batches(sorted(range(32), key=lambda k: lens[probe[k]]), [users[p] for p in probe]):
        run_batch([users[probe[k]] for k in bt], [tasks[probe[k]] for k in bt])
    if DEV == "mps": torch.mps.synchronize()
    return 32 / (time.perf_counter() - t0)
cold = _probe_pass(); rate = _probe_pass(); _clean()
done_rows = sum(len(np.load(f"{SHD}/{f}")["idx"]) for f in os.listdir(SHD) if f.startswith("sh_"))
print(f"probe cold {cold:.2f} | warm {rate:.2f} rows/s -> {N - done_rows:,} rows left -> ~{(N - done_rows)/rate/60:.0f} min")
if rate < MIN_RATE and not FORCE:
    raise RuntimeError(f"{rate:.2f} rows/s is below {MIN_RATE}. Check Activity Monitor (memory pressure / swap), "
                       "quit other apps, restart the kernel. Or set FORCE = True.")

In [ ]:
# ---- full run: sharded, resumable, never silent ----
done = {int(f[3:7]) for f in os.listdir(SHD) if f.startswith("sh_")}
n_sh = int(np.ceil(N / SHARD)); todo = [s for s in range(n_sh) if s not in done]
left = sum(min((s+1)*SHARD, N) - s*SHARD for s in todo)
print(f"shards {n_sh} x {SHARD} | done {len(done)} | to do {len(todo)} ({left:,} rows)")
t_run, n_run, nb = time.perf_counter(), 0, 0
for sh in todo:
    idx = np.arange(sh*SHARD, min((sh+1)*SHARD, N))
    U, T = [users[i] for i in idx], [tasks[i] for i in idx]
    A = {li: np.empty((len(idx), HID), np.float32) for li in LAYERS}
    E = {li: np.empty((len(idx), HID), np.float32) for li in LAYERS}
    order = sorted(range(len(idx)), key=lambda k: _ntok(U[k]))
    t_sh, n_sh_done, win_t, win_n = time.perf_counter(), 0, time.perf_counter(), 0
    for bt in _batches(order, U):
        out = run_batch([U[k] for k in bt], [T[k] for k in bt])
        for li in LAYERS: A[li][bt], E[li][bt] = out[li]
        nb += 1; n_sh_done += len(bt); n_run += len(bt); win_n += len(bt)
        _clean(deep=(nb % DEEP_CLEAN_EVERY == 0))
        if nb % PROGRESS_EVERY == 0:
            r = win_n / max(time.perf_counter() - win_t, 1e-9)
            print(f"    shard {sh:03d} {n_sh_done:4d}/{len(idx)}  {r:5.2f} rows/s | eta {(left-n_run)/max(r,1e-9)/60:6.1f} min", flush=True)
            win_t, win_n = time.perf_counter(), 0
    np.savez_compressed(f"{SHD}/sh_{sh:04d}.npz", idx=idx,
                        **{f"anchor_{li}": A[li] for li in LAYERS}, **{f"end_{li}": E[li] for li in LAYERS})
    print(f"  shard {sh:03d} saved {len(idx)} rows {len(idx)/(time.perf_counter()-t_sh):5.2f} rows/s", flush=True)
print("extraction complete" if n_run or not todo else "nothing to do")

In [ ]:
# ---- assemble ----
files = sorted(f for f in os.listdir(SHD) if f.startswith("sh_"))
for li in LAYERS:
    A = np.full((N, HID), np.nan, np.float32); E_ = np.full((N, HID), np.nan, np.float32)
    for f in files:
        d = np.load(f"{SHD}/{f}"); A[d["idx"]] = d[f"anchor_{li}"]; E_[d["idx"]] = d[f"end_{li}"]
    assert np.isfinite(A).all() and np.isfinite(E_).all(), f"layer {li}: rows missing -- re-run the extraction cell"
    np.save(f"{OUT}/fresh_end_{li}.npy", E_); np.save(f"{OUT}/fresh_anchor_{li}.npy", A)
    del A, E_
json.dump(dict(model=MODEL_ID, layers=LAYERS, system_prompt=SYSTEM_PROMPT, max_len=MAX_LEN, fresh=N,
               anchor="last token of task (v19b rule)"), open(f"{OUT}/manifest.json", "w"), indent=2)
for h in HANDLES: h.remove()
del model, base; gc.collect()
if DEV == "mps": torch.mps.empty_cache()
print(f"wrote {OUT}/fresh_* ({N:,} rows); model released.\nNEXT: Kernel -> Restart, then v23c2_guards.ipynb.")